# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SrijanKumar123/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
%pip -q install duckdb
import duckdb
from google.colab import userdata

token = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute(f"""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN '{token}'
    )
""")

In [2]:
REL = """
read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

In [3]:
%pip -q install -U duckdb huggingface_hub

import duckdb
from google.colab import userdata
from huggingface_hub import whoami

token = userdata.get("HF_TOKEN")

print("Token found:", token is not None)
print("Logged in as:", whoami(token=token)["name"])

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 46.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 kB 20.2 MB/s eta 0:00:00
Token found: True
Logged in as: srijan317


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Logisitic Regression is chosen because it provides direct linear coefficient weights, allowing us to see exactly how features like search position or impression volume impact prediction log-odds.  Logistic Regression serves as the direct, honest baseline model.  

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import numpy as np
import pandas as pd

dataset = con.sql(f""" SELECT client_hash_id, gsc_avg_position, gsc_impressions, ga4_pageviews, ga4_sessions, gsc_clicks
FROM {REL} """).df()
dataset["ctr"] = dataset["gsc_clicks"] / dataset["gsc_impressions"].replace(
    0, np.nan
)

dataset["is_striking_distance"] = (
    (dataset["gsc_avg_position"] > 3) &
     (dataset["gsc_avg_position"] <= 20) &
 (dataset["gsc_impressions"] >= 500)
 ).astype(int)

dataset["ctr"] = dataset["ctr"].fillna(0)
dataset["ga4_pageviews"] = dataset["ga4_pageviews"].fillna(0)
dataset["ga4_sessions"] = dataset["ga4_sessions"].fillna(0)
dataset["gsc_avg_position"] = dataset["gsc_avg_position"].fillna(100.0)

y = dataset["is_striking_distance"]
X = dataset[["gsc_avg_position", "gsc_impressions", "ctr",  "ga4_pageviews", "ga4_sessions"]]

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Grouping by client domain guarantees that all performance records belonging to a specific client stay strictly within either the training set or the validation set.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import GroupKFold

groups = dataset["client_hash_id"]
gkf = GroupKFold(n_splits=4)


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.